# Adversarial filter (o3)

Every implicit-citation candidate produced by the bi-encoder is adjudicated by **o3** as an adversarial filter: does the excerpt implicitly apply the rule of the proposed article (OUI/NON + rationale)? Runs as OpenAI batch jobs, merges the answers, completes any gaps, and analyses the resulting validated dataset.

## Prepare candidates

Loads the deduplicated bi-encoder predictions and adds empty `pred_API` / `justification_API` columns for the batch annotations.

In [ ]:
import pandas as pd
import os

# Load the deduplicated inference predictions and add empty annotation columns
df = pd.read_excel("artifacts/inference/output_predictions_unique.xlsx")  # not shipped — regenerated by this step (see DATA.md)

for col in ["decision_id", "chunk_id", "pred_art"]:
    if col in df.columns:
        df[col] = df[col].astype(str)

df["pred_API"] = pd.NA
df["justification_API"] = pd.NA

os.makedirs("artifacts/validation", exist_ok=True)
df.to_excel("artifacts/validation/candidates.xlsx", index=False)
df.to_parquet("artifacts/validation/candidates.parquet", index=False)
print(f"{len(df):,} candidate pairs written to artifacts/validation/candidates.xlsx")

## Build the o3 batch helpers

Reusable helpers to write batch requests, submit/track a batch, download its output, and merge OUI/NON answers back into the candidate table.

In [ ]:
import os
import json
import re
import pandas as pd
import openai

# o3 is asked to judge whether a decision excerpt *implicitly* applies the rule of a
# given article (without citing its number). Strict OUI/NON + one-sentence rationale.
SYSTEM_PROMPT = (
    "Tu es un juriste assistant spécialisé en droit français."
    "Ta tâche consiste à déterminer si un extrait de décision judiciaire met en œuvre, applique ou reprend de manière implicite la règle de droit d'un article de loi donné, "
    "c'est-à-dire sans que le numéro ou la référence de l'article ne soit explicitement cité dans l'extrait, "
    "mais en reprenant son contenu, sa règle ou son principe."
    "Réponds strictement au format suivant :"
    "\nOUI. [Justification d'une ou deux phrases.]\n"
    "ou\n"
    "NON. [Justification d'une ou deux phrases.]\n"
    "Si tu n'es pas sûr ou qu'il y a un doute, réponds NON."
    "Ne fais jamais de réponse mitigée ou d'hypothèse."
    "Utilise uniquement les éléments du texte pour juger si la règle de droit de l'article semble appliquée ou reprise dans l'extrait, même sans citation explicite."
)


def write_batch_requests(df, out_jsonl, max_completion_tokens=1024):
    """Write one OpenAI batch request per (chunk, article) pair to a JSONL file."""
    os.makedirs(os.path.dirname(out_jsonl), exist_ok=True)
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            user_prompt = (
                f"### Extrait de la d\u00e9cision :\n{row['text']}\n\n"
                f"### Article propos\u00e9 :\n{row['article_text']}\n\n"
                "La r\u00e8gle de droit de cet article est-elle appliqu\u00e9e ou reprise de fa\u00e7on "
                "implicite dans cet extrait, sans que le num\u00e9ro d'article soit cit\u00e9 ? "
                "R\u00e9ponds uniquement par OUI ou NON, selon le format ci-dessus."
            )
            request = {
                "custom_id": f"{row['decision_id']}|{row['chunk_id']}|{row['pred_art']}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "o3",
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user_prompt},
                    ],
                    "max_completion_tokens": max_completion_tokens,
                },
            }
            f.write(json.dumps(request, ensure_ascii=False) + "\n")
    print(f"Batch requests written to: {out_jsonl}")


def submit_batch(jsonl_path):
    """Upload a JSONL request file and launch a 24h OpenAI batch. Returns the batch id."""
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    file_resp = client.files.create(file=open(jsonl_path, "rb"), purpose="batch")
    batch = client.batches.create(
        input_file_id=file_resp.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    print("input_file_id =", file_resp.id, "| batch id =", batch.id)
    return batch.id


def download_batch_output(output_file_id, out_path):
    """Download a completed batch's output file to out_path (JSONL)."""
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    with open(out_path, "wb") as f:
        f.write(client.files.content(output_file_id).read())
    print(f"Batch output downloaded: {out_path}")


def _parse_custom_id(custom_id):
    parts = str(custom_id).strip().split("|")
    return (parts[0], parts[1], parts[2]) if len(parts) >= 3 else ("", "", "")


def merge_batch_output(df_src, jsonl_path):
    """Parse a batch output JSONL (OUI/NON + rationale) and fill empty annotations in df_src."""
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            out = json.loads(line)
            decision_id, chunk_id, pred_art = _parse_custom_id(out.get("custom_id", ""))
            try:
                content = out["response"]["body"]["choices"][0]["message"]["content"]
            except Exception:
                content = ""
            m = re.match(r"^(OUI|NON)\. ?(.*)", content.strip(), re.DOTALL)
            pred_api, justif_api = (m.group(1), m.group(2).strip()) if m else ("", "")
            rows.append({"decision_id": decision_id, "chunk_id": chunk_id, "pred_art": pred_art,
                         "pred_API": pred_api, "justification_API": justif_api})
    df_preds = pd.DataFrame(rows)

    for col in ["decision_id", "chunk_id", "pred_art"]:
        df_src[col] = df_src[col].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
        df_preds[col] = df_preds[col].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

    mask_empty = df_src["pred_API"].isna() | (df_src["pred_API"].astype(str).str.strip() == "")
    batch_dict = df_preds.set_index(["decision_id", "chunk_id", "pred_art"]).to_dict(orient="index")

    def fill(row):
        if mask_empty.loc[row.name]:
            info = batch_dict.get((row["decision_id"], row["chunk_id"], row["pred_art"]))
            if info:
                row["pred_API"] = info["pred_API"]
                row["justification_API"] = info["justification_API"]
        return row

    df_out = df_src.apply(fill, axis=1)
    n_ann = (df_out["pred_API"].notna() & (df_out["pred_API"].astype(str).str.strip() != "")).sum()
    print(f"Annotated: {n_ann} / {len(df_out)} ({n_ann / len(df_out):.2%})")
    return df_out

## Run the full batch

Judges every candidate pair with o3.

In [ ]:
# --- Full batch: judge every candidate pair with o3 -------------------------
candidates = pd.read_excel("artifacts/validation/candidates.xlsx")
write_batch_requests(candidates, "artifacts/validation/batch_full/batch_requests.jsonl")

# Submit the batch (needs OPENAI_API_KEY). Then poll until it completes:
# batch_id = submit_batch("artifacts/validation/batch_full/batch_requests.jsonl")
# client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
# print(client.batches.retrieve(batch_id).status)

# Once completed, download the output file (paste its output_file_id):
# download_batch_output("file-xxxxxxxx", "artifacts/validation/batch_full/batch_output.jsonl")

## Merge batch results

In [ ]:
# Merge the full-batch answers back into the candidate table
df_src = pd.read_excel("artifacts/validation/candidates.xlsx")
df_merged = merge_batch_output(df_src, "artifacts/validation/batch_full/batch_output.jsonl")
df_merged.to_excel("artifacts/validation/full_batch_merged.xlsx", index=False)

## Complete missing annotations

Re-queries only the pairs o3 left unanswered (larger token budget), then merges them into the final validated dataset.

In [ ]:
# --- Completion batch: re-query only the pairs o3 left unanswered ------------
df = pd.read_excel("artifacts/validation/full_batch_merged.xlsx")
mask_todo = df["pred_API"].isna() | (df["pred_API"].astype(str).str.strip() == "")
df_todo = df[mask_todo].copy()
print(f"{len(df_todo)} entries to re-batch out of {len(df)}")

write_batch_requests(df_todo, "artifacts/validation/batch_completion/batch_requests.jsonl",
                     max_completion_tokens=2048)

# batch_id = submit_batch("artifacts/validation/batch_completion/batch_requests.jsonl")
# download_batch_output("file-xxxxxxxx", "artifacts/validation/batch_completion/batch_output.jsonl")

In [ ]:
# Fill the remaining gaps and export the final validated dataset
df_src = pd.read_excel("artifacts/validation/full_batch_merged.xlsx")
df_final = merge_batch_output(df_src, "artifacts/validation/batch_completion/batch_output.jsonl")
df_final.to_excel("artifacts/validation/full_preds_o3_final.xlsx", index=False)
print("Final validated dataset: artifacts/validation/full_preds_o3_final.xlsx")

## Analyse the final validated dataset

OUI/NON breakdown, OUI rate by FAISS distance, ROC of distance vs. o3 label, and export of the accepted (OUI) pairs.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("artifacts/validation/full_preds_o3_final.xlsx")  # not shipped — regenerated by this step (see DATA.md)
pred_clean = df["pred_API"].astype(str).str.strip().str.upper()

nb_total = len(df)
nb_oui = (pred_clean == "OUI").sum()
nb_non = (pred_clean == "NON").sum()
total_pred_art = df["pred_art"].nunique()
n_art_oui = df.loc[pred_clean == "OUI", "pred_art"].nunique()

print(f"Total entries: {nb_total}")
print(f"OUI: {nb_oui} ({nb_oui / nb_total:.2%}) | NON: {nb_non} ({nb_non / nb_total:.2%})")
print(f"Unique articles overall: {total_pred_art} | with at least one OUI: {n_art_oui}")

In [ ]:
# OUI rate by FAISS-distance bin
df["distance"] = pd.to_numeric(df["distance"], errors="coerce")
bins = np.arange(0, 1.01, 0.1)
labels = [f"[{bins[i]:.1f}-{bins[i+1]:.1f}[" for i in range(len(bins) - 1)]
df["distance_bin"] = pd.cut(df["distance"], bins=bins, labels=labels, right=False, include_lowest=True)

summary = df.groupby("distance_bin", observed=True)["pred_API"].agg(
    total="count",
    nb_oui=lambda x: (x.astype(str).str.strip().str.upper() == "OUI").sum(),
).reset_index()
summary["pct_oui"] = 100 * summary["nb_oui"] / summary["total"]
print(summary[["distance_bin", "nb_oui", "total", "pct_oui"]])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Invert distance so higher score = better match
scores = 1 - df["distance"].fillna(0)
y_true = (df["pred_API"].astype(str).str.upper() == "OUI").astype(int)

# ROC curve
fpr, tpr, thresholds = roc_curve(y_true, scores)
auc = roc_auc_score(y_true, scores)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})", color="royalblue")
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)

# Annotate selected thresholds for readability
step = max(1, len(thresholds) // 10)
for i in range(0, len(thresholds), step):
    plt.annotate(f"{thresholds[i]:.2f}",
        (fpr[i], tpr[i]),
        textcoords="offset points",
        xytext=(5, -5),
        fontsize=8,
        color="darkred")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve with thresholds (scores = 1 - distance)")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Export the accepted (OUI) pairs, sorted by ascending distance
df_oui = df[pred_clean == "OUI"].copy()
df_oui["distance"] = pd.to_numeric(df_oui["distance"], errors="coerce")
df_oui = df_oui.sort_values("distance", ascending=True)
df_oui.to_excel("artifacts/validation/final_oui.xlsx", index=False)
print(f"final_oui.xlsx written with {len(df_oui)} accepted pairs")